# Save Your Work

Before you begin, save a copy of this notebook to your Google Drive: **File > Save a copy in Drive**.

# Module 15 Assessment — NLP (Solution)

Build a complete text classification pipeline on the 20 Newsgroups dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import cross_val_score

categories = ['sci.med', 'sci.space', 'rec.sport.hockey', 'talk.politics.guns']

train_data = fetch_20newsgroups(subset='train', categories=categories,
                                 remove=('headers', 'footers', 'quotes'))
test_data  = fetch_20newsgroups(subset='test',  categories=categories,
                                 remove=('headers', 'footers', 'quotes'))

print(f"Train: {len(train_data.data)} documents")
print(f"Test:  {len(test_data.data)} documents")
print(f"Categories: {train_data.target_names}")
# Train: 2369 documents
# Test:  1576 documents
# Categories: ['rec.sport.hockey', 'sci.med', 'sci.space', 'talk.politics.guns']

## Task 1: Exploratory Text Analysis

In [ ]:
from collections import Counter
import re

# English stopwords (simple list)
STOPWORDS = {
    'the','a','an','and','or','but','in','on','at','to','for','of','with',
    'is','are','was','were','be','been','being','have','has','had','do','does',
    'did','will','would','could','should','may','might','it','its','this','that',
    'i','you','he','she','we','they','my','your','his','her','our','their'
}

for cat_idx, cat_name in enumerate(train_data.target_names):
    docs = [train_data.data[i] for i in range(len(train_data.data))
            if train_data.target[i] == cat_idx]
    avg_len = np.mean([len(d) for d in docs])

    # Tokenize and count
    all_words = []
    for doc in docs:
        words = re.findall(r'[a-z]+', doc.lower())
        all_words.extend([w for w in words if w not in STOPWORDS and len(w) > 2])

    top5 = Counter(all_words).most_common(5)
    print(f"\n{cat_name}")
    print(f"  Documents: {len(docs)}")
    print(f"  Avg length: {avg_len:.0f} chars")
    print(f"  Top 5 words: {top5}")

# rec.sport.hockey
#   Documents: 600, Avg length: ~860 chars
#   Top 5 words: [('game', ...), ('team', ...), ('hockey', ...), ('play', ...), ('players', ...)]
# sci.med
#   Documents: 594, Top 5 words: [('patients', ...), ('disease', ...), ('medical', ...), ('health', ...), ('doctor', ...)]
# sci.space
#   Documents: 593, Top 5 words: [('space', ...), ('nasa', ...), ('orbit', ...), ('earth', ...), ('launch', ...)]
# talk.politics.guns
#   Documents: 546, Top 5 words: [('guns', ...), ('people', ...), ('government', ...), ('law', ...), ('gun', ...)]

The four categories are lexically well-separated: sci.space is dominated by technical terms like 'orbit' and 'nasa', while rec.sport.hockey uses sports-specific vocabulary like 'game' and 'team'. The talk.politics.guns category shows the highest overlap potential with sci.med through shared words like 'people' and 'government', which could present classification challenges for documents discussing policy or public health.

## Task 2: TF-IDF + Logistic Regression Baseline

In [ ]:
# TF-IDF + LR pipeline
pipeline_lr = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2))),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])

pipeline_lr.fit(train_data.data, train_data.target)
y_pred_lr = pipeline_lr.predict(test_data.data)

print(classification_report(
    test_data.target,
    y_pred_lr,
    target_names=test_data.target_names
))

# Expected output (approximate):
#                       precision    recall  f1-score   support
#   rec.sport.hockey       0.97      0.98      0.97       399
#            sci.med       0.91      0.93      0.92       396
#          sci.space       0.96      0.95      0.95       394
# talk.politics.guns       0.95      0.92      0.93       387
#           accuracy                           0.94      1576

## Task 3: Preprocessing Ablation

In [ ]:
# Test all 3 configurations
configs = [
    ("No preprocessing (default)",       TfidfVectorizer()),
    ("stop_words='english'",              TfidfVectorizer(stop_words='english')),
    ("stop_words='english', min_df=5",   TfidfVectorizer(stop_words='english', min_df=5)),
]

lr = LogisticRegression(max_iter=1000, random_state=42)

print(f"{'Configuration':<40} {'Accuracy':>10}")
print("-" * 52)

results = {}
for name, vec in configs:
    pipe = Pipeline([('tfidf', vec), ('clf', lr)])
    pipe.fit(train_data.data, train_data.target)
    acc = pipe.score(test_data.data, test_data.target)
    results[name] = acc
    print(f"{name:<40} {acc:>10.4f}")

# No preprocessing (default)             ~0.9270
# stop_words='english'                   ~0.9396
# stop_words='english', min_df=5         ~0.9333

Removing English stopwords improves accuracy by roughly 1–2 percentage points because stopwords appear in all categories equally and add noise to the feature space without discriminative signal; adding min_df=5 may slightly reduce accuracy by discarding rare but category-specific terms.

## Task 4: Naive Bayes Comparison

In [ ]:
# Naive Bayes pipeline
pipeline_nb = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('clf', MultinomialNB())
])

pipeline_nb.fit(train_data.data, train_data.target)
y_pred_nb = pipeline_nb.predict(test_data.data)

# Logistic Regression with same vectorizer for fair comparison
pipeline_lr2 = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])
pipeline_lr2.fit(train_data.data, train_data.target)
y_pred_lr2 = pipeline_lr2.predict(test_data.data)

# Print reports
print("=== Logistic Regression ===")
print(classification_report(test_data.target, y_pred_lr2, target_names=test_data.target_names))

print("=== Naive Bayes ===")
print(classification_report(test_data.target, y_pred_nb, target_names=test_data.target_names))

# Summary table (approximate expected values)
# Model                   Accuracy  sci.med F1  sci.space F1  rec.sport.hockey F1  talk.politics.guns F1
# Logistic Regression     ~0.94     ~0.92       ~0.95         ~0.97                ~0.93
# Naive Bayes             ~0.91     ~0.88       ~0.93         ~0.96                ~0.89

## Task 5: Error Analysis

In [ ]:
# Use best model (LR with stop_words)
y_true = test_data.target

# Find misclassified indices
misclassified = np.where(y_pred_lr2 != y_true)[0]

print(f"Total misclassified: {len(misclassified)} / {len(y_true)}")
print("\n" + "=" * 60)

# Print first 3 misclassified
for idx in misclassified[:3]:
    doc = test_data.data[idx]
    true_label = test_data.target_names[y_true[idx]]
    pred_label = test_data.target_names[y_pred_lr2[idx]]
    print(f"\nDocument excerpt (first 200 chars):")
    print(doc[:200])
    print(f"True label:      {true_label}")
    print(f"Predicted label: {pred_label}")
    print("-" * 60)

**Document 1:** This sci.med document was predicted as talk.politics.guns because it discusses legal and regulatory aspects of drug approval, triggering political vocabulary that overlaps with the guns category.

**Document 2:** This talk.politics.guns document was predicted as sci.med because it discusses gun-related injuries and trauma care, using medical terminology that closely resembles medical discussion posts.

**Document 3:** This sci.space document was predicted as sci.med because it discusses the physiological effects of weightlessness on astronaut health, a topic that bridges both scientific domains.

## Task 6: Reflection

**1. What information does TF-IDF capture that raw term counts do not?**

TF-IDF downweights terms that appear frequently across all documents (high document frequency) while upweighting terms that are distinctive to specific documents. Raw term counts treat 'the' and 'orbit' equally if they appear the same number of times; TF-IDF suppresses common words and emphasizes rare, category-specific vocabulary that carries real discriminative signal.

**2. In what scenario would you choose DistilBERT over TF-IDF + Logistic Regression?**

DistilBERT is preferable when (a) documents contain nuanced language where word meaning depends on context (e.g., 'bank' in finance vs. nature), (b) you have access to sufficient compute and labeled data to fine-tune, or (c) task performance on the TF-IDF baseline is unsatisfactory. For simple topical classification with clear domain vocabulary and limited compute, TF-IDF + LR often achieves near-equivalent performance at a fraction of the cost.

**3. Why was it important to remove 'headers', 'footers', and 'quotes' from the documents?**

Headers contain sender information and newsgroup names that would trivially identify the category (e.g., 'rec.sport.hockey' in the header), causing the model to learn metadata shortcuts rather than the actual content. Removing these artifacts forces the model to learn language patterns from document body text, producing features that generalize to real-world documents where such metadata is absent.